# Gupapuyŋu TTS — Piper training on Kaggle (multi-speaker)

Resumes a single-speaker checkpoint (from `train_kaggle.ipynb`) as a multi-speaker model. The training data is **not** distributed with this model; upload your own as a Kaggle dataset named `gup-multi` containing `gup_multi.zip`:

```
dataset_multi/
  metadata.csv        # one line per utterance:  id|speaker|text   (UTF-8, no header)
  wavs/<id>.wav       # 22050 Hz, mono, 16-bit PCM
espeak-gup/
  lang-gup
  gup_rules
  gup_list
```

`speaker` is any label (e.g. `mali.wata`); it becomes a key of `speaker_id_map` in the exported `.json`. Future releases of this model will be trained on fully synthetic data so that the training set itself can be published; the current training set is withheld for privacy reasons.

**Session options:** GPU T4 x2 (or P100), Internet ON. **Inputs:** `gup-multi` (required); `gup-checkpoint` with the single-speaker `gup-checkpoint.tar` (first cycle) or `gup-multi-checkpoint` with `gup-multi-checkpoint.tar` (later cycles). Everything bulky goes to `/tmp`; `/kaggle/working` keeps only checkpoints and outputs.

Outputs: `gup-multi-checkpoint.tar`, `gup_multi.onnx` + `gup_multi.onnx.json`.

In [ ]:
!nvidia-smi -L
!ls /kaggle/input
!df -h /kaggle/working /tmp

In [ ]:
%%bash
set -e
T=/tmp/gup
mkdir -p $T
DF=$(find /kaggle/input -maxdepth 5 -type d -name dataset_multi 2>/dev/null | head -1)
EG=$(find /kaggle/input -maxdepth 5 -type d -name espeak-gup 2>/dev/null | head -1)
if [ -z "$DF" ]; then
  ZIP=$(find /kaggle/input -maxdepth 5 -name gup_multi.zip 2>/dev/null | head -1)
  test -n "$ZIP" || { echo 'ERROR: gup_multi.zip not found in /kaggle/input'; exit 1; }
  test -d $T/dataset_multi || unzip -q -o "$ZIP" -d $T/
  DF=$T/dataset_multi
  EG=$T/espeak-gup
fi
test -f "$DF/metadata.csv" || { echo "ERROR: $DF/metadata.csv missing"; exit 1; }
test -f "$EG/lang-gup" || { echo "ERROR: $EG/lang-gup missing"; exit 1; }
ln -sfn "$DF" $T/dataset_multi_link
ln -sfn "$EG" $T/espeak_gup_link
ls "$DF/wavs" | wc -l
cut -d'|' -f2 "$DF/metadata.csv" | sort | uniq -c
df -h /tmp | tail -1

In [ ]:
%%bash
set -e
apt-get install -y -q espeak-ng > /dev/null 2>&1
DATA=$(espeak-ng --version | sed -n 's/.*Data at: *//p')
mkdir -p "$DATA/lang/aus"
cp /tmp/gup/espeak_gup_link/lang-gup "$DATA/lang/aus/gup"
grep -q '^apostrophe' "$DATA/lang/aus/gup" || echo "apostrophe 2" >> "$DATA/lang/aus/gup"
cd /tmp/gup/espeak_gup_link && espeak-ng --compile=gup
espeak-ng -v gup -q --ipa "Ŋama' marrtjina Galiwin'kulili"

In [ ]:
%%bash
set -e
T=/tmp/gup
test -d $T/piper || git clone -q https://github.com/rhasspy/piper.git $T/piper
pip install -q uv
VENV_OK=0
if [ -x $T/venv/bin/python ]; then
  $T/venv/bin/python -c "import sys; assert sys.version_info[:2]==(3,10)" 2>/dev/null && VENV_OK=1
fi
if [ "$VENV_OK" = 0 ]; then
  rm -rf $T/venv
  uv python install 3.10
  uv venv --seed --python 3.10 $T/venv
fi
source $T/venv/bin/activate
python3 -c "import sys; assert sys.version_info[:2]==(3,10), sys.version"
pip install -q "pip<24.1" wheel "setuptools<70"
cd $T/piper/src/python
pip install -q -e .
pip install -q "torchmetrics==0.11.4" "numpy<2" six "tensorboard==2.11.2" "protobuf==3.20.3"
bash build_monotonic_align.sh > /dev/null
python3 -c "import torch, pytorch_lightning, piper_phonemize; print('torch', torch.__version__, '| lightning', pytorch_lightning.__version__, '| cuda:', torch.cuda.is_available())"

In [ ]:
%%bash
set -e
T=/tmp/gup
EG=$T/espeak_gup_link
ESDATA=$(espeak-ng --version | sed -n 's/.*Data at: *//p')
test -f "$ESDATA/gup_dict" || (cd "$EG" && espeak-ng --compile=gup > /dev/null 2>&1)
source $T/venv/bin/activate
PHDATA=$(python3 -c "import piper_phonemize, pathlib; print(pathlib.Path(piper_phonemize.__file__).parent / 'espeak-ng-data')")
mkdir -p "$PHDATA/lang/aus"
cp "$ESDATA/lang/aus/gup" "$PHDATA/lang/aus/gup"
cp "$ESDATA/gup_dict" "$PHDATA/gup_dict"
python3 -c "from piper_phonemize import phonemize_espeak; print(phonemize_espeak(\"Ŋama' marrtjina Galiwin'kulili\", 'gup'))"

In [ ]:
%%bash
set -e
T=/tmp/gup
source $T/venv/bin/activate
cd $T/piper/src/python
test -f $T/training/config.json || python3 -m piper_train.preprocess \
  --language gup \
  --input-dir $T/dataset_multi_link \
  --output-dir $T/training \
  --dataset-format ljspeech \
  --sample-rate 22050
python3 -c "import json; c=json.load(open('$T/training/config.json')); print('speakers:', c['num_speakers'], c['speaker_id_map'])"
df -h /tmp /kaggle/working | tail -2

In [ ]:
%%bash
set -e
T=/tmp/gup
CKPT=$(find /kaggle/input -name '*.ckpt' 2>/dev/null | head -1)
MODE=multi
if [ -z "$CKPT" ]; then
  TAR=$(find /kaggle/input -name '*.tar' 2>/dev/null | head -1)
  if [ -n "$TAR" ]; then
    mkdir -p $T/ckpt
    tar -xf "$TAR" -C $T/ckpt
    CKPT=$(find $T/ckpt -name '*.ckpt' | head -1)
    case "$(basename "$TAR")" in *multi*) MODE=multi ;; *) MODE=single ;; esac
  fi
fi
if [ -z "$CKPT" ]; then
  mkdir -p $T/ckpt
  wget -q -O $T/ckpt/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt"
  CKPT=$T/ckpt/base.ckpt
  MODE=single
fi
echo "$CKPT" > $T/ckpt_initial.txt
echo "$MODE" > $T/ckpt_mode.txt
echo "resuming from: $CKPT ($MODE)"

In [ ]:
%%bash
T=/tmp/gup
source $T/venv/bin/activate
CKPT=$(cat $T/ckpt_initial.txt)
MODE=$(cat $T/ckpt_mode.txt)
LAST=$(ls -t /kaggle/working/gup_voice/lightning_logs/version_*/checkpoints/*.ckpt 2>/dev/null | head -1)
test -n "$LAST" && { CKPT="$LAST"; MODE=multi; }
if [ "$MODE" = single ]; then RESUME="--resume_from_single_speaker_checkpoint"; else RESUME="--resume_from_checkpoint"; fi
echo "training from: $CKPT ($RESUME)"
mkdir -p /kaggle/working/gup_voice
cd $T/piper/src/python
export PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:128
timeout --signal=INT --kill-after=300 18000 python3 -m piper_train \
  --dataset-dir $T/training \
  --accelerator gpu \
  --devices 1 \
  --batch-size 8 \
  --max-phoneme-ids 400 \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --max_epochs 3200 \
  --max_time 00:04:45:00 \
  $RESUME "$CKPT" \
  --checkpoint-epochs 5 \
  --precision 32 \
  --default_root_dir /kaggle/working/gup_voice
echo "training finished (code $?)"
LAST=$(ls -t /kaggle/working/gup_voice/lightning_logs/version_*/checkpoints/*.ckpt 2>/dev/null | head -1)
if [ -n "$LAST" ]; then
  cp "$LAST" /kaggle/working/resume.ckpt
  tar -cf /kaggle/working/gup-multi-checkpoint.tar -C /kaggle/working resume.ckpt
  echo "checkpoint saved: $(basename "$LAST") -> gup-multi-checkpoint.tar"
fi
df -h /kaggle/working | tail -1

In [ ]:
%%bash
T=/tmp/gup
source $T/venv/bin/activate
cd $T/piper/src/python
test -f /kaggle/working/resume.ckpt || { echo "nothing to export"; exit 0; }
timeout 900 python3 -m piper_train.export_onnx /kaggle/working/resume.ckpt /kaggle/working/gup_multi.onnx \
  && cp $T/training/config.json /kaggle/working/gup_multi.onnx.json \
  || echo "onnx export failed or timed out; the checkpoint is intact"
ls -lh /kaggle/working/gup_multi.onnx* 2>/dev/null
rm -rf /kaggle/working/gup_voice /kaggle/working/resume.ckpt
du -sh /kaggle/working